# Analisis Dataset Stunting

**Proyek:** Prediksi Stunting pada Balita
**Dataset:** stunting.csv
**Deskripsi:** Dataset berisi data tinggi badan balita berdasarkan umur (bulan) dan jenis kelamin, dengan status stunting (Normal / Stunted / Severely Stunted).

---
## Tahapan Analisis
1. **Setup & Data Loading** - Load dataset dan eksplorasi awal
2. **Data Preprocessing** - Pembersihan data, fixing inkonsistensi
3. **Handling Missing Values** - Deteksi dan penanganan nilai hilang
4. **Outlier Detection & Handling** - Deteksi dan penanganan outlier
5. **Export Clean Dataset** - Simpan dataset bersih dan ringkasan
---

---
# BAGIAN 1: SETUP & DATA LOADING
---

In [ ]:
# ============================================================
# BAGIAN 1: SETUP & DATA LOADING
# Tujuan: Import library, load dataset, dan eksplorasi awal
# ============================================================

# 1.1 Import Library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

# Buat folder processed jika belum ada
os.makedirs('../data/processed', exist_ok=True)

warnings.filterwarnings('ignore')

# Konfigurasi visualisasi
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print("Library berhasil diimport.")

In [ ]:
# 1.2 Load Dataset
df_raw = pd.read_csv('../data/raw/stunting.csv', sep=';')

print(f"Dataset berhasil dimuat!")
print(f"Shape: {df_raw.shape[0]} baris, {df_raw.shape[1]} kolom")

In [ ]:
# 1.3 Eksplorasi Awal
print("=== 5 Data Pertama ===")
display(df_raw.head())

print("\n=== Informasi Dataset ===")
df_raw.info()

print("\n=== Statistik Deskriptif ===")
display(df_raw.describe())

In [ ]:
# 1.4 Distribusi Target (Status)
print("=== Distribusi Status Stunting ===")
status_counts = df_raw['Status'].value_counts()
print(status_counts)
print(f"\nPersentase:")
print(df_raw['Status'].value_counts(normalize=True).mul(100).round(2))

# Visualisasi
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

status_counts.plot(kind='bar', ax=axes[0], color=['green', 'orange', 'red'])
axes[0].set_title('Distribusi Status Stunting')
axes[0].set_xlabel('Status')
axes[0].set_ylabel('Jumlah')
axes[0].tick_params(axis='x', rotation=0)

status_counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                   colors=['green', 'orange', 'red'], startangle=90)
axes[1].set_title('Proporsi Status Stunting')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('../data/processed/distribusi_status_awal.png', dpi=100, bbox_inches='tight')
plt.show()
print("\nVisualisasi tersimpan: data/processed/distribusi_status_awal.png")

In [ ]:
# 1.5 Export CSV - Data Awal & Statistik
# Simpan data mentah untuk referensi
df_raw.to_csv('../data/processed/stunting_00_raw.csv', index=False)
print("Raw data exported: data/processed/stunting_00_raw.csv")

# Simpan ringkasan statistik
summary_stats = df_raw.describe(include='all').round(2)
summary_stats.to_csv('../data/processed/stunting_00_summary_stats.csv')
print("Summary stats exported: data/processed/stunting_00_summary_stats.csv")

# Simpan distribusi status
dist_status = df_raw['Status'].value_counts().reset_index()
dist_status.columns = ['Status', 'Jumlah']
dist_status['Persentase'] = (dist_status['Jumlah'] / dist_status['Jumlah'].sum() * 100).round(2)
dist_status.to_csv('../data/processed/stunting_00_distribusi_status.csv', index=False)
print("Distribusi status exported: data/processed/stunting_00_distribusi_status.csv")

print(f"\nTotal {df_raw.shape[0]} sampel, {df_raw.shape[1]} kolom siap diproses.")

> **Kesimpulan Bagian 1:** Dataset berhasil dimuat. Terlihat ada 1.351 sampel dengan 4 kolom. Distribusi status menunjukkan ketidakseimbangan kelas (class imbalance) dengan mayoritas Normal.

---
# BAGIAN 2: DATA PREPROCESSING
---

In [ ]:
# ============================================================
# BAGIAN 2: DATA PREPROCESSING
# Tujuan: Membersihkan dataset dari inkonsistensi
# ============================================================

# Buat salinan untuk preprocessing
df = df_raw.copy()

# 2.1 Standardisasi Nama Kolom (lowercase, tanpa spasi)
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print("Nama kolom setelah standardisasi:")
print(df.columns.tolist())

In [ ]:
# 2.2 Cek dan Fix Inkonsistensi String
print("=== Nilai Unik di Kolom 'status' SEBELUM pembersihan ===")
print(df['status'].unique())
print(df['status'].value_counts())

# Fix: hapus spasi berlebih, standardisasi kapitalisasi
df['status'] = df['status'].str.strip().str.title()

print("\n=== Nilai Unik di Kolom 'status' SETELAH pembersihan ===")
print(df['status'].unique())
print(df['status'].value_counts())

In [ ]:
# 2.3 Standardisasi Kolom 'jk' (Jenis Kelamin)
print("=== Nilai Unik di Kolom 'jk' SEBELUM ===")
print(df['jk'].unique())
print(df['jk'].value_counts())

# Standardisasi: L -> Laki-Laki, P -> Perempuan
jk_mapping = {'L': 'Laki-Laki', 'P': 'Perempuan'}
df['jk'] = df['jk'].str.strip().map(jk_mapping)

print("\n=== Nilai Unik di Kolom 'jk' SETELAH ===")
print(df['jk'].unique())
print(df['jk'].value_counts())

In [ ]:
# 2.4 Deteksi Duplikasi Data
print("=== Deteksi Duplikasi Data ===")
duplicates = df.duplicated().sum()
print(f"Jumlah data duplikat: {duplicates}")

if duplicates > 0:
    print("\nBaris duplikat:")
    display(df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(10))
    # Hapus duplikat
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"\n{duplicates} duplikat telah dihapus.")
    print(f"Shape setelah hapus duplikat: {df.shape}")
else:
    print("Tidak ada data duplikat.")

In [ ]:
# 2.5 Cek dan Konversi Tipe Data
print("=== Tipe Data Sebelum Konversi ===")
print(df.dtypes)

# Konversi umur_bulan dan tinggi ke numerik (jika ada yang object)
df['umur_bulan'] = pd.to_numeric(df['umur_bulan'], errors='coerce')
df['tinggi'] = pd.to_numeric(df['tinggi'], errors='coerce')

print("\n=== Tipe Data Setelah Konversi ===")
print(df.dtypes)

print("\n=== Ringkasan Setelah Preprocessing ===")
display(df.head())
print(f"Shape: {df.shape}")

In [ ]:
# 2.6 Export Intermediate CSV - Hasil Preprocessing
df.to_csv('../data/processed/stunting_01_preprocessed.csv', index=False)
print("Data preprocessing selesai!")
print(f"Intermediate CSV tersimpan: data/processed/stunting_01_preprocessed.csv")
print(f"Shape: {df.shape[0]} baris, {df.shape[1]} kolom")

> **Kesimpulan Bagian 2:** Data preprocessing selesai. Inkonsistensi string telah diperbaiki (spasi pada "Stunted", standardisasi kolom 'jk'). Tipe data sudah sesuai.

---
# BAGIAN 3: HANDLING MISSING VALUES
---

In [ ]:
# ============================================================
# BAGIAN 3: HANDLING MISSING VALUES
# Tujuan: Deteksi dan penanganan missing values
# ============================================================

# 3.1 Deteksi Missing Values
print("=== Jumlah Missing Values per Kolom ===")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Jumlah Missing': missing, 'Persentase (%)': missing_pct.round(2)})
print(missing_df[missing_df['Jumlah Missing'] > 0] if missing.sum() > 0 else "Tidak ada missing values!")

print(f"\nTotal missing values: {df.isnull().sum().sum()}")

In [ ]:
# 3.2 Visualisasi Missing Values
if df.isnull().sum().sum() > 0:
    plt.figure(figsize=(8, 4))
    sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
    plt.title('Peta Missing Values')
    plt.tight_layout()
    plt.show()
else:
    print("Tidak ada missing values untuk divisualisasikan. Dataset sudah bersih dari nilai hilang.")

In [ ]:
# 3.3 Penanganan Missing Values (Jika Ada)
total_missing = df.isnull().sum().sum()

if total_missing > 0:
    # Strategi:
    # - Kolom numerik (umur_bulan, tinggi): isi dengan median
    # - Kolom kategorikal (jk, status): isi dengan modus

    for col in df.columns:
        missing_count = df[col].isnull().sum()
        if missing_count > 0:
            if df[col].dtype in ['float64', 'int64']:
                median_val = df[col].median()
                df[col] = df[col].fillna(median_val)
                print(f"Kolom '{col}': {missing_count} missing diisi dengan median ({median_val:.2f})")
            else:
                modus_val = df[col].mode()[0]
                df[col] = df[col].fillna(modus_val)
                print(f"Kolom '{col}': {missing_count} missing diisi dengan modus ({modus_val})")

    print(f"\nTotal {total_missing} missing values telah ditangani.")
    print(f"Missing values setelah penanganan: {df.isnull().sum().sum()}")
else:
    print("Tidak ada missing values. Dataset sudah lengkap.")
    print("Langkah ini tetap dipertahankan untuk mengantisipasi data baru di masa depan.")

# Export Intermediate CSV - Setelah Handling Missing Values
df.to_csv('../data/processed/stunting_02_no_missing.csv', index=False)
print("\nIntermediate CSV tersimpan: data/processed/stunting_02_no_missing.csv")
print(f"Shape: {df.shape[0]} baris, {df.shape[1]} kolom")

> **Kesimpulan Bagian 3:** Dataset stunting tidak memiliki missing values. Semua data lengkap dan siap untuk tahap selanjutnya.

---
*Bersambung ke Bagian 4: Outlier Detection & Handling...*